# 03 Pipeline RAG Complet

## Objectif
Assembler le pipeline RAG complet avec stratégie de fallback :
- **Mode strict (Prompt avec instructions strictes)** : réponse basée uniquement sur le contexte PubMed
- **Mode fallback (Prompt avec chain-of-thought)** : chain-of-thought quand le contexte est insuffisant

## Architecture
Question → ChromaDB → Score pertinence

↙          ↘

< 0.80           ≥ 0.80
(suffisant)     (insuffisant)

↓               ↓

Mode strict    Mode fallback

↘              ↙

Réponse + Sources + Niveau de confiance

## Modèle LLM
**Llama 3.3 70B** via API Groq. On a décidé de choisi ce modèle car il est gratuit, ultra-rapide et a une bonne réputation en science médical.

In [1]:
# ============================================================
# 03 PIPELINE RAG COMPLET
# ============================================================

import os
import time
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

# LangChain
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_chroma import Chroma
from langchain_groq import ChatGroq
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough

# Chargement de la clé API
env_path = Path('..') / '.env'
with open(env_path, 'r', encoding='ascii') as f:
    for line in f:
        line = line.strip()
        if '=' in line and not line.startswith('#'):
            key, value = line.split('=', 1)
            os.environ[key.strip()] = value.strip()

GROQ_API_KEY = os.environ.get("GROQ_API_KEY")

print(" Configuration chargée")
print(f"Clé Groq : {'' if GROQ_API_KEY else '❌'}")

# Chargement du modèle d'embedding
print("\nChargement du modèle d'embedding...")
embedding_model = HuggingFaceEmbeddings(
    model_name    = "all-MiniLM-L6-v2",
    model_kwargs  = {"device": "cpu"},
    encode_kwargs = {"normalize_embeddings": True}
)

# Chargement de la base ChromaDB
print("Chargement de la base ChromaDB...")
vectorstore = Chroma(
    persist_directory = "../data/processed/chroma_db",
    embedding_function= embedding_model,
    collection_name   = "pubmed_biomedical"
)

print(f" Base ChromaDB chargée : {vectorstore._collection.count():,} chunks")

# Initialisation du LLM
llm = ChatGroq(
    model       = "llama-3.3-70b-versatile",
    temperature = 0.1,  # Faible température pour des réponses factuelles
    api_key     = GROQ_API_KEY
)

print(f" LLM chargé : Llama 3.3 70B via Groq")

 Configuration chargée
Clé Groq : 

Chargement du modèle d'embedding...


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 6512.70it/s]


Chargement de la base ChromaDB...
 Base ChromaDB chargée : 836 chunks
 LLM chargé : Llama 3.3 70B via Groq


## Définition des prompts

### Prompt Mode Strict
Le LLM répond UNIQUEMENT à partir du contexte PubMed fourni. Il cite ses sources et indique un niveau de confiance.

### Prompt Mode Fallback 
Activé quand le contexte est insuffisant (score ≥ 0.80). Le LLM raisonne étape par étape et avertit l'utilisateur que la réponse n'est pas basée sur des sources directes.

In [2]:
# ============================================================
# Définition des prompts
# ============================================================

# --- PROMPT MODE STRICT ---
PROMPT_STRICT = ChatPromptTemplate.from_messages([
    ("system", """You are a specialized biomedical research assistant 
focused on mental health of young graduates and unemployment.

STRICT RULES:
1. Answer ONLY based on the provided PubMed context below
2. NEVER invent information not present in the context
3. Always cite the PMID and year of the sources you use
4. If the context partially answers the question, say so explicitly
5. Use clear, professional medical language
6. Structure your answer with: Main Finding, Evidence, and Limitations

CONTEXT FROM PUBMED:
{context}
"""),
    ("human", """Question: {question}

Please provide a evidence-based answer citing the relevant PubMed articles.""")
])

# --- PROMPT MODE FALLBACK ---
PROMPT_FALLBACK = ChatPromptTemplate.from_messages([
    ("system", """You are a specialized biomedical research assistant 
focused on mental health of young graduates and unemployment.

IMPORTANT: The available PubMed context has limited relevance 
to this question. You will use chain-of-thought reasoning based on 
general biomedical knowledge, but you MUST clearly warn the user.

RULES FOR FALLBACK MODE:
1. Start with: "⚠️ LIMITED CONTEXT WARNING: ..."
2. Use step-by-step reasoning (chain-of-thought)
3. Clearly distinguish between what comes from context vs general knowledge
4. Recommend the user consult additional specialized sources
5. Be conservative, prefer acknowledging uncertainty over speculation

AVAILABLE CONTEXT (limited relevance):
{context}
"""),
    ("human", """Question: {question}

Please reason step-by-step and provide the best possible answer 
given the limited context, with appropriate caveats.""")
])

print(" Prompts définis")
print(f"  - Mode Strict  : réponse basée uniquement sur le contexte")
print(f"  - Mode Fallback: chain-of-thought avec avertissement")

 Prompts définis
  - Mode Strict  : réponse basée uniquement sur le contexte
  - Mode Fallback: chain-of-thought avec avertissement


## Fonction RAG avec Fallback

La fonction principale du système :
1. Recherche les K chunks les plus pertinents dans ChromaDB
2. Évalue le score de pertinence moyen
3. Route vers le bon prompt (strict ou fallback)
4. Génère la réponse avec le LLM
5. Retourne la réponse + sources + métadonnées

In [3]:
# ============================================================
# Fonction RAG avec Fallback
# ============================================================

SEUIL_PERTINENCE = 0.80  # Score au-dessus duquel on bascule en fallback
K_CHUNKS         = 5     # Nombre de chunks à récupérer

def formater_contexte(docs_scores):
    """Formate les chunks récupérés en contexte lisible pour le LLM."""
    contexte_parts = []
    for i, (doc, score) in enumerate(docs_scores):
        part = f"""
[Source {i+1}]
PMID    : {doc.metadata['pmid']}
Titre   : {doc.metadata['titre']}
Journal : {doc.metadata['journal']}
Année   : {doc.metadata['annee']}
Score   : {score:.4f}
Contenu : {doc.page_content}
"""
        contexte_parts.append(part)
    return "\n---\n".join(contexte_parts)

def extraire_sources(docs_scores):
    """Extrait les métadonnées des sources pour la réponse finale."""
    sources = []
    for doc, score in docs_scores:
        sources.append({
            "pmid"      : doc.metadata['pmid'],
            "titre"     : doc.metadata['titre'],
            "journal"   : doc.metadata['journal'],
            "annee"     : doc.metadata['annee'],
            "score"     : round(score, 4),
            "url_pubmed": f"https://pubmed.ncbi.nlm.nih.gov/{doc.metadata['pmid']}/"
        })
    return sources

def rag_query(question, k=K_CHUNKS, seuil=SEUIL_PERTINENCE, verbose=True):
    """
    Pipeline RAG complet avec fallback.
    
    Args:
        question : Question de l'utilisateur
        k        : Nombre de chunks à récupérer
        seuil    : Seuil de pertinence pour le fallback
        verbose  : Afficher les détails du processus
    
    Returns:
        dict avec réponse, sources, mode utilisé, score moyen
    """
    
    if verbose:
        print(f"❓ Question : {question}")
        print(f"{'─'*60}")
    
    # ── Étape 1 : Recherche dans ChromaDB ──
    docs_scores = vectorstore.similarity_search_with_score(
        query=question, k=k
    )
    
    # ── Étape 2 : Évaluation de la pertinence ──
    scores      = [score for _, score in docs_scores]
    score_moyen = sum(scores) / len(scores)
    score_min   = min(scores)
    
    if verbose:
        print(f" Scores de similarité :")
        print(f"   Score moyen : {score_moyen:.4f}")
        print(f"   Score min   : {score_min:.4f}")
        print(f"   Seuil       : {seuil}")
    
    # ── Étape 3 : Choix du mode ──
    if score_moyen < seuil:
        mode   = "strict"
        prompt = PROMPT_STRICT
        if verbose:
            print(f"   Mode        : STRICT (contexte suffisant)")
    else:
        mode   = "fallback"
        prompt = PROMPT_FALLBACK
        if verbose:
            print(f"   Mode        : FALLBACK (contexte insuffisant)")
    
    # ── Étape 4 : Formatage du contexte ──
    contexte = formater_contexte(docs_scores)
    sources  = extraire_sources(docs_scores)
    
    # ── Étape 5 : Génération de la réponse ──
    if verbose:
        print(f"\n Génération de la réponse...")
    
    chain    = prompt | llm | StrOutputParser()
    reponse  = chain.invoke({
        "question": question,
        "context" : contexte
    })
    
    # ── Étape 6 : Résultat final ──
    resultat = {
        "question"    : question,
        "reponse"     : reponse,
        "mode"        : mode,
        "score_moyen" : round(score_moyen, 4),
        "score_min"   : round(score_min, 4),
        "sources"     : sources,
        "n_chunks"    : len(docs_scores)
    }
    
    return resultat

print(" Fonction RAG définie")
print(f"   Seuil fallback : {SEUIL_PERTINENCE}")
print(f"   K chunks       : {K_CHUNKS}")

 Fonction RAG définie
   Seuil fallback : 0.8
   K chunks       : 5


## Tests du Pipeline RAG

On teste le système avec 4 questions :
1. Question directement dans notre domaine → Mode Strict attendu
2. Question sur un sous-thème spécifique → Mode Strict attendu
3. Question hors domaine → Mode Fallback attendu
4. Question complexe sur notre méta-analyse → Mode Strict attendu

In [4]:
# ============================================================
# Test 1 Question directement dans notre domaine
# ============================================================

resultat_1 = rag_query(
    question="What are the main mental health consequences of unemployment in young university graduates?",
    verbose=True
)

print(f"\n{'='*60}")
print(" RÉPONSE :")
print(f"{'='*60}")
print(resultat_1['reponse'])

print(f"\n{'='*60}")
print(" SOURCES UTILISÉES :")
print(f"{'='*60}")
for i, source in enumerate(resultat_1['sources']):
    print(f"\n[{i+1}] PMID {source['pmid']} ({source['annee']})")
    print(f"     {source['titre'][:80]}...")
    print(f"     {source['url_pubmed']}")
    print(f"     Score : {source['score']}")

❓ Question : What are the main mental health consequences of unemployment in young university graduates?
────────────────────────────────────────────────────────────
 Scores de similarité :
   Score moyen : 0.6706
   Score min   : 0.6214
   Seuil       : 0.8
   Mode        : STRICT (contexte suffisant)

 Génération de la réponse...

 RÉPONSE :
**Main Finding**: Unemployment in young university graduates is associated with significant mental health consequences, including increased risks of anxiety, depression, psychological distress, and poor mental well-being.

**Evidence**: Studies have consistently shown that economic downturns and unemployment can have a scarring effect on the mental health of young adults. A longitudinal study (PMID: 33359536, 2021) found that graduating during a recession was associated with increased risks of high psychological distress, diagnoses of depression or anxiety, and lower levels of social functioning and mental well-being among young adults, particula

On observe que pour ce test le Mode Strict à été activé avec un score moyen 0.67 < seuil 0.80.

La qualité de la réponse est intéressante avec:

- Une structure parfaite : Main Finding → Evidence → Limitations.
- Les sources citées avec PMID et année.
- Liens PubMed cliquables.
- Le LLM a correctement identifié les limites des études (focus sur les hommes, petit échantillon)
- La réponse est nuancée et professionnelle.

Sources récupérées :

- Articles de 2008 à 2024 ce qui est une bonne couverture temporelle.
- Journaux reconnus (Annals of Epidemiology, etc.)

In [5]:
# ============================================================
# Test 2 Question sur un sous-thème spécifique
# ============================================================

resultat_2 = rag_query(
    question="What is the role of precarious employment and job insecurity on psychological distress among recent graduates?",
    verbose=True
)

print(f"\n{'='*60}")
print(" RÉPONSE :")
print(f"{'='*60}")
print(resultat_2['reponse'])

print(f"\n{'='*60}")
print(" SOURCES UTILISÉES :")
print(f"{'='*60}")
for i, source in enumerate(resultat_2['sources']):
    print(f"\n[{i+1}] PMID {source['pmid']} ({source['annee']})")
    print(f"     {source['titre'][:80]}...")
    print(f"     {source['url_pubmed']}")

❓ Question : What is the role of precarious employment and job insecurity on psychological distress among recent graduates?
────────────────────────────────────────────────────────────
 Scores de similarité :
   Score moyen : 0.7680
   Score min   : 0.7324
   Seuil       : 0.8
   Mode        : STRICT (contexte suffisant)

 Génération de la réponse...

 RÉPONSE :
**Main Finding:** Precarious employment and job insecurity play a significant role in contributing to psychological distress among recent graduates.

**Evidence:** Studies have consistently shown that precarious employment and job insecurity are associated with increased psychological distress, including anxiety, depression, and stress-related disorders. A register-linked cohort study (PMID: 37567755, 2023) found that precarious employment in early adulthood was associated with an increased risk of later mental health problems, including depression, anxiety, and stress-related disorders, with a hazard ratio (HR) of 1.51 (95% CI

On voir un voir que la qualité a monté d'un cran. Le Mode Strict est activé encore avec un score moyen 0.768 < seuil 0.80, mais on est proche du seuil, ce qui signifie que la question est plus spécifique.

La qualité de la réponse est bien:

- Le LLM cite des chiffres précis avec un Hazard Ratio 1.51 (95% CI 1.42-1.60) extrait directement de l'étude.
- 5 sources différentes de 2019 à 2024, encore une bonne diversité temporelle.
- Identification des limites méthodologiques (biais de self-report, généralisation géographique).
- Recommandation politique en fin de réponse, le niveau d'analyse est très professionnel.

In [6]:
# ============================================================
# Test 3 Question hors domaine → Fallback attendu
# ============================================================

resultat_3 = rag_query(
    question="What are the latest treatments for cardiovascular diseases in elderly patients?",
    verbose=True
)

print(f"\n{'='*60}")
print(" RÉPONSE :")
print(f"{'='*60}")
print(resultat_3['reponse'])

❓ Question : What are the latest treatments for cardiovascular diseases in elderly patients?
────────────────────────────────────────────────────────────
 Scores de similarité :
   Score moyen : 1.3379
   Score min   : 1.1671
   Seuil       : 0.8
   Mode        : FALLBACK (contexte insuffisant)

 Génération de la réponse...

 RÉPONSE :
⚠️ LIMITED CONTEXT WARNING: The available context has limited relevance to the question about the latest treatments for cardiovascular diseases in elderly patients. The provided sources primarily focus on health-related quality of life, employment status, and medication non-adherence in various patient populations, rather than specifically addressing treatments for cardiovascular diseases in the elderly.

1. **Understanding the Question**: The question asks for the latest treatments for cardiovascular diseases in elderly patients. This requires information on current medical practices, pharmaceutical interventions, and possibly lifestyle modifications sp

C'était attentu, le Mode Fallback a été activé avec un score moyen 1.34 >> seuil 0.80, la question est clairement hors domaine.

Le système se Comporte bien :

- Avertissement clair dès le début ⚠️
- Raisonnement chain-of-thought en 5 étapes numérotées.
- Honnêteté sur les limites, "this is speculative".
- Recommandation de consulter des sources spécialisées (AHA, ESC).
- N'invente aucune information médicale.

C'est exactement le comportement qu'on voulait pour une application médicale ie ne jamais halluciner, toujours avertir.

In [7]:
# ============================================================
# Test 4 Question complexe liée à la méta-analyse
# ============================================================

resultat_4 = rag_query(
    question="What does the systematic review evidence say about the causal relationship between post-graduation unemployment and onset of depression and anxiety disorders in young adults?",
    verbose=True
)

print(f"\n{'='*60}")
print(" RÉPONSE :")
print(f"{'='*60}")
print(resultat_4['reponse'])

print(f"\n{'='*60}")
print(" SOURCES UTILISÉES :")
print(f"{'='*60}")
for i, source in enumerate(resultat_4['sources']):
    print(f"\n[{i+1}] PMID {source['pmid']} ({source['annee']})")
    print(f"     {source['titre'][:80]}...")
    print(f"     {source['url_pubmed']}")

❓ Question : What does the systematic review evidence say about the causal relationship between post-graduation unemployment and onset of depression and anxiety disorders in young adults?
────────────────────────────────────────────────────────────
 Scores de similarité :
   Score moyen : 0.7236
   Score min   : 0.6566
   Seuil       : 0.8
   Mode        : STRICT (contexte suffisant)

 Génération de la réponse...

 RÉPONSE :
Main Finding: The systematic review evidence suggests an association between post-graduation unemployment and mental health, including depression and anxiety disorders, in young adults. However, the causal relationship between the two is less clear.

Evidence: A systematic review (PMID: 31291827, 2020) analyzed 17 articles and found that while there is an association between unemployment among young people and mental health, the evidence for a causal relationship is mixed. Cross-sectional studies (N=5) showed an association between unemployment and mental health, b

Mode Strict activé avec un score moyen 0.72 < seuil 0.80.

La qualité scientifique est remarquable :

- Le LLM a trouvé et cité une vraie revue systématique (PMID 31291827, 2020), c'est ce qu'on cherchait.
- Il distingue correctement association vs causalité, une nuance fondamentale en épidémiologie.
- Il mentionne le problème des confondeurs et du mental health at baseline.
- Données longitudinales citées (effets à 21, 30 et 42 ans).
- Les limites méthodologiques identifiées sont exactement celles d'une vraie méta-analyse.

Cette réponse pourrait littéralement servir dans ma future méta-analyse (Impact de la transition post-diplôme et du chômage initial sur la santé mentale des jeunes diplômés universitaires : une revue systématique et méta-analyse d'études observationnelles) comme point de départ de revue de littérature. 

In [8]:
# ============================================================
# Sauvegarde des résultats de test
# ============================================================

import json

resultats_tests = {
    "test_1_unemployment_mental_health"    : {
        "question"    : resultat_1['question'],
        "mode"        : resultat_1['mode'],
        "score_moyen" : resultat_1['score_moyen'],
        "n_sources"   : len(resultat_1['sources'])
    },
    "test_2_precarious_employment"         : {
        "question"    : resultat_2['question'],
        "mode"        : resultat_2['mode'],
        "score_moyen" : resultat_2['score_moyen'],
        "n_sources"   : len(resultat_2['sources'])
    },
    "test_3_cardiovascular_hors_domaine"   : {
        "question"    : resultat_3['question'],
        "mode"        : resultat_3['mode'],
        "score_moyen" : resultat_3['score_moyen'],
        "n_sources"   : len(resultat_3['sources'])
    },
    "test_4_systematic_review_causality"   : {
        "question"    : resultat_4['question'],
        "mode"        : resultat_4['mode'],
        "score_moyen" : resultat_4['score_moyen'],
        "n_sources"   : len(resultat_4['sources'])
    }
}

# Sauvegarde
output_path = Path('../data/processed/resultats_tests_rag.json')
with open(output_path, 'w', encoding='utf-8') as f:
    json.dump(resultats_tests, f, indent=2, ensure_ascii=False)

print(" Résultats sauvegardés : data/processed/resultats_tests_rag.json")
print(f"\n=== RÉCAPITULATIF ===")
for nom, res in resultats_tests.items():
    print(f"\n{nom}")
    print(f"  Mode        : {res['mode']}")
    print(f"  Score moyen : {res['score_moyen']}")
    print(f"  Sources     : {res['n_sources']}")

 Résultats sauvegardés : data/processed/resultats_tests_rag.json

=== RÉCAPITULATIF ===

test_1_unemployment_mental_health
  Mode        : strict
  Score moyen : 0.6706
  Sources     : 5

test_2_precarious_employment
  Mode        : strict
  Score moyen : 0.768
  Sources     : 5

test_3_cardiovascular_hors_domaine
  Mode        : fallback
  Score moyen : 1.3379
  Sources     : 5

test_4_systematic_review_causality
  Mode        : strict
  Score moyen : 0.7236
  Sources     : 5
